# Data Deduplication & Curation

Web-scraped corpora are full of duplicates: the same article on twenty aggregators, the
same licence boilerplate on a million repositories, the same page crawled twice. Training
on them is not neutral — it wastes compute, increases memorisation of the repeated text,
and quietly corrupts your train/test split.

The problem is that exact duplicates are the easy case and the rare one. Most duplication
is *near*-duplication: the same document with a different header, a changed date, one
edited paragraph. Finding those at web scale is what this notebook is about, and the
answer — MinHash with LSH — is a genuinely elegant piece of algorithm design that turns an
`O(n²)` comparison into something linear.

Third topic in the [Pre-training](tokenization-bpe.ipynb) track. Shares its machinery with
[Benchmark Contamination](../12-model-evaluation/benchmark-contamination.ipynb).

## 1. What & Why

**What deduplication buys you**, in rough order of importance:

- **Less memorisation.** The strongest predictor of whether a model regurgitates a
  training string verbatim is how many times it appeared. Deduplication is the most
  effective privacy and copyright-risk control available at the data stage.
- **A valid evaluation.** Duplicates that straddle a train/test split leak the test set.
  A meaningful fraction of the test sets in common corpora appear verbatim in their own
  training splits.
- **Better loss per unit compute.** Deduplicated corpora reach a given loss in fewer
  steps. You are not paying to learn the same boilerplate a thousand times.
- **Less degenerate output.** Heavily duplicated text is over-represented in the model's
  output distribution.

**The scale problem.** Comparing every pair of documents is `O(n²)`. For 10⁹ documents
that is 10¹⁸ comparisons — impossible. Every practical method is a way of avoiding almost
all of those comparisons while still finding almost all of the duplicates.

**The judgement call.** Deduplication is *lossy curation*: some repetition is
legitimate signal. Common phrases should be common. Deciding what counts as a duplicate
is a modelling decision, not a purely technical one.

## 2. Mental Model

**A fingerprint you can compare without the document.**

The core trick, and it is worth understanding rather than memorising:

1. Represent each document as the **set of its shingles** (overlapping n-grams). Document
   similarity becomes set similarity — the Jaccard index.
2. **MinHash**: hash every shingle, keep only the minimum. The probability that two
   documents share the same minimum is *exactly* their Jaccard similarity. Repeat with
   `k` independent hash functions and you have a `k`-number signature whose agreement rate
   estimates Jaccard.
3. **LSH**: split the signature into bands and hash each band. Similar documents collide
   in at least one band with high probability; dissimilar ones almost never do. Only
   compare documents that collided.

The first step is a change of representation, the second is dimensionality reduction with
a provable guarantee, and the third turns comparison into a hash-table lookup.

The single most useful intuition: **`P(min hashes agree) = Jaccard similarity`**. That
identity is why MinHash works at all, and everything else is engineering around it.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Exact dedup** | Hash the whole document (or normalise then hash). Cheap, and catches only identical text. |
| **Shingle / n-gram** | A window of `n` consecutive words or characters. The unit of comparison. |
| **Jaccard similarity** | `\|A ∩ B\| / \|A ∪ B\|` over shingle sets. The standard near-duplicate measure. |
| **MinHash** | The minimum hash value over a set. `P(equal) = Jaccard`. |
| **Signature** | `k` MinHashes per document. Estimates Jaccard to `±1/√k`. |
| **LSH banding** | Split the signature into `b` bands of `r` rows; candidates are pairs colliding in any band. |
| **S-curve** | The banding threshold: `1 − (1 − s^r)^b`. Sharp transition around `s ≈ (1/b)^(1/r)`. |
| **Suffix-array dedup** | Removes repeated *substrings* rather than whole documents. Complementary to MinHash. |
| **Fuzzy vs exact** | Whether near-duplicates count. Almost always yes for web text. |
| **Quality filtering** | The other half of curation: classifiers, perplexity filters, heuristics. Distinct from dedup. |
| **Repetition and epochs** | With limited unique data, repeating up to ~4 epochs costs little; beyond that returns decay sharply. |

## 4. Setup

Standard library plus NumPy. The algorithms here are short, and implementing MinHash is
the fastest way to believe the probability identity it rests on.

In [1]:
# %pip install numpy

import hashlib
import random
import numpy as np
from collections import defaultdict

rng = np.random.default_rng(0)
random.seed(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — exact dedup catches almost nothing

Normalise-and-hash is the first thing everyone reaches for. Here is what it misses.

In [2]:
base = ("the quick brown fox jumps over the lazy dog while the sun sets slowly "
        "behind the distant hills and the river flows quietly toward the sea")

variants = {
    "identical":              base,
    "different whitespace":   base.replace(" ", "  "),
    "different case":         base.upper(),
    "one word changed":       base.replace("quick", "swift"),
    "date header added":      "Published 2024-03-11. " + base,
    "one sentence removed":   base.split(" while ")[0],
    "unrelated document":     "machine learning models require careful evaluation "
                              "and thoughtful dataset construction to be useful",
}

def exact_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:12]

def normalised_hash(text):
    return hashlib.sha256(" ".join(text.lower().split()).encode()).hexdigest()[:12]

h_exact, h_norm = exact_hash(base), normalised_hash(base)
print(f"{'variant':24} {'exact match':>12} {'normalised match':>18}")
for name, text in variants.items():
    print(f"{name:24} {str(exact_hash(text) == h_exact):>12} "
          f"{str(normalised_hash(text) == h_norm):>18}")

print("\nNormalisation buys you whitespace and case, and nothing else. A single changed")
print("word, an added date line, or a trimmed paragraph all defeat it completely --")
print("and those are exactly the transformations that produce real web duplicates.")
print("\nThat is why near-duplicate detection is not an optional refinement.")

variant                   exact match   normalised match
identical                        True               True
different whitespace            False               True
different case                  False               True
one word changed                False              False
date header added               False              False
one sentence removed            False              False
unrelated document              False              False

Normalisation buys you whitespace and case, and nothing else. A single changed
word, an added date line, or a trimmed paragraph all defeat it completely --
and those are exactly the transformations that produce real web duplicates.

That is why near-duplicate detection is not an optional refinement.


### Example 2 — shingles and Jaccard: the right notion of similarity

Turn each document into a set of overlapping n-grams, and similarity becomes a set
operation.

In [3]:
def shingles(text, n=5):
    '''Overlapping n-grams of words. n=5 is a common choice for documents.'''
    words = text.lower().split()
    return {" ".join(words[i:i + n]) for i in range(max(1, len(words) - n + 1))}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

base_sh = shingles(base)
print(f"base document: {len(base_sh)} shingles of 5 words\n")
print(f"{'variant':24} {'shingles':>9} {'shared':>7} {'Jaccard':>9}")
for name, text in variants.items():
    s = shingles(text)
    print(f"{name:24} {len(s):9d} {len(s & base_sh):7d} {jaccard(s, base_sh):9.3f}")

print("\nNow the near-duplicates are visibly near: a changed word or an added header")
print("scores high, and the unrelated document scores 0.")
print("\nNote the effect of shingle size on what counts as similar:")
for n in (3, 5, 8, 12):
    j = jaccard(shingles(base, n), shingles(base.replace("quick", "swift"), n))
    print(f"  n={n:2d}: Jaccard with a one-word edit = {j:.3f}")
print("\nLonger shingles are STRICTER -- one edited word destroys more of them. n=5")
print("is a reasonable default for prose; code and short documents want smaller n.")

base document: 22 shingles of 5 words

variant                   shingles  shared   Jaccard
identical                       22      22     1.000
different whitespace            22      22     1.000
different case                  22      22     1.000
one word changed                22      20     0.833
date header added               24      22     0.917
one sentence removed             5       5     0.227
unrelated document               9       0     0.000

Now the near-duplicates are visibly near: a changed word or an added header
scores high, and the unrelated document scores 0.

Note the effect of shingle size on what counts as similar:
  n= 3: Jaccard with a one-word edit = 0.846
  n= 5: Jaccard with a one-word edit = 0.833
  n= 8: Jaccard with a one-word edit = 0.810
  n=12: Jaccard with a one-word edit = 0.765

Longer shingles are STRICTER -- one edited word destroys more of them. n=5
is a reasonable default for prose; code and short documents want smaller n.


### Example 3 — MinHash: estimating Jaccard without storing the sets

The identity that makes web-scale dedup possible. Verify it empirically before trusting
it.

In [4]:
P = (1 << 31) - 1        # a Mersenne prime; keeps a*h+b inside int64

def minhash_signature(shingle_set, k=128, seed=0):
    '''k independent min-hashes, via the universal family h_i(x) = (a_i*x + b_i) mod P.

    Both documents must use the SAME (a, b) -- that is what makes the signatures
    comparable, and it is why `seed` has to match on both sides.
    '''
    r = np.random.default_rng(seed)
    a = r.integers(1, P, size=k, dtype=np.int64)
    b = r.integers(0, P, size=k, dtype=np.int64)
    if not shingle_set:
        return np.full(k, P, dtype=np.int64)
    hs = np.array([int(hashlib.blake2b(sh.encode(), digest_size=8).hexdigest(), 16) % P
                   for sh in shingle_set], dtype=np.int64)
    return ((a[:, None] * hs[None, :] + b[:, None]) % P).min(axis=1)

def estimate_jaccard(sig_a, sig_b):
    return float(np.mean(sig_a == sig_b))

sig_base = minhash_signature(base_sh)
print(f"{'variant':24} {'true Jaccard':>13} {'MinHash (k=128)':>17} {'error':>8}")
for name, text in variants.items():
    s = shingles(text)
    true = jaccard(s, base_sh)
    est = estimate_jaccard(sig_base, minhash_signature(s))
    print(f"{name:24} {true:13.3f} {est:17.3f} {est - true:+8.3f}")

print("\nThe estimate tracks the true value using only 128 numbers per document,")
print("regardless of how long the documents are. That is the whole point: you can")
print("hold signatures for a billion documents in memory; you cannot hold their")
print("shingle sets.\n")

print("accuracy improves as 1/sqrt(k):")
target = shingles(base.replace("quick", "swift"))
true = jaccard(target, base_sh)
for k in (16, 64, 256, 1024):
    ests = [estimate_jaccard(minhash_signature(base_sh, k, seed=s),
                             minhash_signature(target, k, seed=s)) for s in range(5)]
    print(f"  k={k:5d}: estimates {min(ests):.3f}-{max(ests):.3f}  (true {true:.3f}), "
          f"predicted +/- {1/np.sqrt(k):.3f}")

variant                   true Jaccard   MinHash (k=128)    error
identical                        1.000             1.000   +0.000
different whitespace             1.000             1.000   +0.000
different case                   1.000             1.000   +0.000
one word changed                 0.833             0.805   -0.029
date header added                0.917             0.836   -0.081
one sentence removed             0.227             0.188   -0.040
unrelated document               0.000             0.000   +0.000

The estimate tracks the true value using only 128 numbers per document,
regardless of how long the documents are. That is the whole point: you can
hold signatures for a billion documents in memory; you cannot hold their
shingle sets.

accuracy improves as 1/sqrt(k):
  k=   16: estimates 0.812-0.938  (true 0.833), predicted +/- 0.250
  k=   64: estimates 0.797-0.859  (true 0.833), predicted +/- 0.125
  k=  256: estimates 0.805-0.855  (true 0.833), predicted +/- 0.062


### Example 4 — LSH banding, and tuning the S-curve

MinHash still requires comparing every pair. LSH avoids that: split the signature into
bands, hash each band, and only compare documents that land in the same bucket. The
band/row split gives you a tunable threshold.

In [5]:
def lsh_probability(s, b, r):
    '''P(two documents with similarity s become candidates) = 1 - (1 - s^r)^b'''
    return 1 - (1 - s ** r) ** b

configs = [(32, 4), (16, 8), (8, 16), (4, 32)]      # (bands, rows); b*r = 128
print("P(becoming a candidate pair) as a function of true similarity:\n")
print(f"{'similarity':>11} " + " ".join(f"{f'b={b},r={r}':>12}" for b, r in configs))
for s in (0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95):
    print(f"{s:11.2f} " + " ".join(f"{lsh_probability(s, b, r):12.3f}"
                                   for b, r in configs))

print("\napproximate threshold (the steep part of the S-curve), (1/b)^(1/r):")
for b, r in configs:
    print(f"  b={b:2d}, r={r:2d}  ->  threshold ~ {(1/b) ** (1/r):.2f}")

print("\nThis is the knob. More bands (fewer rows each) means a LOWER threshold: you")
print("catch more near-duplicates and do more candidate comparisons. Fewer, longer")
print("bands means a higher threshold and a cheaper, stricter run.")
print("\nThe S-curve is what makes LSH work: between s=0.4 and s=0.8 the b=16,r=8")
print("configuration goes from almost never to almost always producing a candidate,")
print("so you get a near-clean separation without comparing everything.")

P(becoming a candidate pair) as a function of true similarity:

 similarity     b=32,r=4     b=16,r=8     b=8,r=16     b=4,r=32
       0.20        0.050        0.000        0.000        0.000
       0.40        0.564        0.010        0.000        0.000
       0.50        0.873        0.061        0.000        0.000
       0.60        0.988        0.237        0.002        0.000
       0.70        1.000        0.613        0.026        0.000
       0.80        1.000        0.947        0.204        0.003
       0.90        1.000        1.000        0.806        0.130
       0.95        1.000        1.000        0.990        0.577

approximate threshold (the steep part of the S-curve), (1/b)^(1/r):
  b=32, r= 4  ->  threshold ~ 0.42
  b=16, r= 8  ->  threshold ~ 0.71
  b= 8, r=16  ->  threshold ~ 0.88
  b= 4, r=32  ->  threshold ~ 0.96

This is the knob. More bands (fewer rows each) means a LOWER threshold: you
catch more near-duplicates and do more candidate comparisons. Fewer, longe

### Example 5 — the whole pipeline, measured against ground truth

A corpus with known duplicates. Run MinHash + LSH end to end, and measure both what it
found and how much work it avoided.

In [6]:
def make_corpus(n_unique=300, n_dupes=120, seed=0):
    r = random.Random(seed)
    vocab = [f"word{i}" for i in range(600)]
    docs, truth = [], {}
    for i in range(n_unique):
        docs.append(" ".join(r.choice(vocab) for _ in range(80)))
    for j in range(n_dupes):
        src = r.randrange(n_unique)
        words = docs[src].split()
        for _ in range(r.randrange(1, 4)):        # a few random edits
            words[r.randrange(len(words))] = r.choice(vocab)
        truth[len(docs)] = src
        docs.append(" ".join(words))
    return docs, truth

docs, truth = make_corpus()
sigs = [minhash_signature(shingles(d), k=128) for d in docs]

def lsh_candidates(sigs, b=16, r=8):
    buckets = defaultdict(list)
    for idx, sig in enumerate(sigs):
        for band in range(b):
            key = (band, tuple(sig[band * r:(band + 1) * r]))
            buckets[key].append(idx)
    pairs = set()
    for members in buckets.values():
        if len(members) > 1:
            for i in range(len(members)):
                for j in range(i + 1, len(members)):
                    pairs.add((members[i], members[j]))
    return pairs

candidates = lsh_candidates(sigs, b=32, r=4)   # lower threshold: favour recall
all_pairs = len(docs) * (len(docs) - 1) // 2

THRESH = 0.4
confirmed = {(i, j) for i, j in candidates
             if estimate_jaccard(sigs[i], sigs[j]) >= THRESH}

found = {(min(d, s), max(d, s)) for d, s in truth.items()}
hits = len(confirmed & found)

print(f"corpus              : {len(docs)} documents ({len(truth)} are near-duplicates)")
print(f"all possible pairs  : {all_pairs:,}")
print(f"LSH candidate pairs : {len(candidates):,}  "
      f"({100*len(candidates)/all_pairs:.2f}% of all pairs)")
print(f"confirmed duplicates: {len(confirmed)}")
print(f"\nrecall   : {hits}/{len(found)} = {hits/len(found):.1%}")
print(f"precision: {hits}/{len(confirmed) if confirmed else 1} = "
      f"{hits/max(1, len(confirmed)):.1%}")
print(f"\nwork avoided: {100*(1 - len(candidates)/all_pairs):.2f}% of pairwise")
print("comparisons never happened. That ratio IMPROVES with corpus size -- pairs grow")
print("quadratically while candidates grow roughly linearly, which is what makes this")
print("viable at web scale.")
print("\nRecall is perfect here only because the configuration was chosen permissively.")
print("LSH is probabilistic and its threshold is a tuning decision -- the SAME corpus")
print("and signatures, under the band splits from Example 4:")
for b, r_ in [(32, 4), (16, 8), (8, 16)]:
    cand = lsh_candidates(sigs, b=b, r=r_)
    conf = {(i, j) for i, j in cand if estimate_jaccard(sigs[i], sigs[j]) >= THRESH}
    h = len(conf & found)
    print(f"  b={b:2d},r={r_:2d}: candidates {len(cand):5d}  recall {h/len(found):5.1%}  "
          f"precision {h/max(1,len(conf)):5.1%}")
print("\nrecall collapses as the threshold rises, and precision rises with it. There is")
print("no setting that maximises both, which is why production pipelines run a")
print("PERMISSIVE LSH pass (high recall, some false candidates) and then verify the")
print("surviving candidates exactly -- cheap, because there are so few of them.")

corpus              : 420 documents (120 are near-duplicates)
all possible pairs  : 87,990
LSH candidate pairs : 138  (0.16% of all pairs)
confirmed duplicates: 138

recall   : 120/120 = 100.0%
precision: 120/138 = 87.0%

work avoided: 99.84% of pairwise
comparisons never happened. That ratio IMPROVES with corpus size -- pairs grow
quadratically while candidates grow roughly linearly, which is what makes this
viable at web scale.

Recall is perfect here only because the configuration was chosen permissively.
LSH is probabilistic and its threshold is a tuning decision -- the SAME corpus
and signatures, under the band splits from Example 4:
  b=32,r= 4: candidates   138  recall 100.0%  precision 87.0%
  b=16,r= 8: candidates   110  recall 86.7%  precision 94.5%
  b= 8,r=16: candidates    37  recall 30.8%  precision 100.0%

recall collapses as the threshold rises, and precision rises with it. There is
no setting that maximises both, which is why production pipelines run a
PERMISSIVE LSH p

## 6. Gotchas & Pitfalls

- **Deduplicating only exactly.** Example 1. Exact hashing finds a small minority of real
  duplicates.
- **Deduplicating the training set but not against the eval set.** The leak that matters
  most. Always dedup train *against* every benchmark you intend to use — see
  [Benchmark Contamination](../12-model-evaluation/benchmark-contamination.ipynb).
- **Shingle size chosen without thought.** Example 2: too small and unrelated documents
  look similar, too large and a single edit destroys the match. Tune it on your data.
- **Removing all repetition.** Some repetition is signal: common phrases *should* be
  common, and a few epochs of repeated data is nearly as good as fresh data. Dedup removes
  *documents*, not *language*.
- **Ignoring which copy you keep.** Duplicate clusters are not interchangeable — one copy
  may be cleaner, better formatted, or from a more reliable source. Keeping a random
  member wastes an easy quality gain.
- **Forgetting substring duplication.** MinHash finds similar *documents*. A long document
  containing a widely-repeated licence block is not a near-duplicate of anything, but the
  block still gets memorised. Suffix-array methods target this.
- **Trusting a single threshold across document lengths.** Short documents have few
  shingles, so their Jaccard estimates are noisy and their similarities are erratic.
- **Doing dedup after quality filtering.** Filtering first means you spend compute
  filtering many copies of the same document.
- **Assuming dedup is quality control.** They are different jobs: deduplication removes
  repetition, quality filtering removes rubbish. A corpus can be perfectly deduplicated
  and entirely worthless.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Web-scale corpus, whole-document duplicates | **MinHash + LSH** (Examples 3–5) |
| Repeated substrings inside otherwise distinct documents | **Suffix array / ExactSubstr** dedup |
| Small corpus (< ~10⁵ docs) | Just compute all pairwise Jaccard — no need for LSH |
| Semantic duplication (same content, rewritten) | Embedding similarity + ANN index. Slower, catches more |
| Checking a benchmark against training data | n-gram overlap — see [Benchmark Contamination](../12-model-evaluation/benchmark-contamination.ipynb) |
| Removing low-quality text | Quality classifiers and perplexity filters — a different problem |

**The honest position.** MinHash + LSH is the right default and has been for twenty-five
years: it is simple, provably approximates Jaccard, parallelises trivially, and its
failure modes are well understood. Embedding-based deduplication catches semantic
duplicates that MinHash cannot, at considerably higher cost, and is usually worth applying
only to a subset.

The most valuable thing here is not the algorithm but the discipline: **dedup before you
train, dedup against your evals, and record what you removed.** The cost of getting it
wrong is not a slightly worse model — it is a model that memorises more than it should and
an evaluation that cannot be trusted.

## 8. Resources

- [Deduplicating Training Data Makes Language Models Better](https://arxiv.org/abs/2107.06499) — Lee et al.; the headline result, and the source of the train/test overlap findings.
- [Deduplicating Training Data Mitigates Privacy Risks in Language Models](https://arxiv.org/abs/2202.06539) — the memorisation-frequency relationship that makes dedup a privacy control.
- [On the Resemblance and Containment of Documents](https://ieeexplore.ieee.org/document/666900) — Broder, 1997. The original MinHash paper.
- [Mining of Massive Datasets, Chapter 3](http://www.mmds.org/) — Leskovec, Rajaraman & Ullman; the clearest textbook treatment of shingling, MinHash and the LSH S-curve.
- [The RefinedWeb Dataset for Falcon LLM](https://arxiv.org/abs/2306.01116) — a production pipeline described in enough detail to reproduce, including dedup thresholds.
- [The Pile](https://arxiv.org/abs/2101.00027) and [Dolma](https://arxiv.org/abs/2402.00159) — two openly-documented corpora; Dolma's toolkit implements exactly this pipeline.
- [Scaling Data-Constrained Language Models](https://arxiv.org/abs/2305.16264) — how many epochs of repeated data are worth having, once you have deduplicated.